# Import Libraries

Enable automatic reloading of modules during interactive development:

In [1]:
%load_ext autoreload
# Changes to all modules will automatically be applied when any cell runs. 
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

import statsmodels.api as sm
from statsmodels.api import Logit

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)

from split_tools import get_train_test

In [3]:
target = 'splashing'
target_set = {'splashing', 'no_fragmentation'}

train, test = get_train_test(
    dataset_filename='df_dimless',
    target=target,
    path_data=Path('..', 'data'),
    target_set=target_set
)

In [4]:
def add_sign(row):
    if row['particle_liquid_density_ratio'] < 1.:
        row['sedimentation_Re'] *= -1
    return row

train = train.apply(
    add_sign,
    axis=1
)
test = test.apply(
    add_sign,
    axis=1
)

In [5]:
def log10(feature):
    return np.log10(feature)

train['relative_roughness'] = np.log10(train['relative_roughness'])
test['relative_roughness'] = np.log10(test['relative_roughness'])

In [6]:
drop_features = [
    'Re', 
    'We', 
    'init_volume_fraction',
    # 'volume_fraction', 
    # 'sedimentation_Re', 
    # 'relative_roughness', 
    # 'inclination',
    # 'wettability',
    'particle_liquid_density_ratio',
    'particle_droplet_diameter_ratio',
    
]

for df in [train, test]:
    df.drop(drop_features, inplace=True, axis=1)

In [7]:
minmax_features = [
    'inclination',
    'volume_fraction',
    # 'relative_roughness'
]

minmax_neg_features = [
    'wettability',
]

def drop_feature(features, drop_features):
    for drop in drop_features:
        if drop in features:
            features.remove(drop)
        
drop_feature(minmax_features, drop_features)
drop_feature(minmax_neg_features, drop_features)

# std_features = list(
#     (
#         set(train.columns)
#         - set(minmax_features)
#         - set(minmax_neg_features)
#         - {target}
#     )
# )

std_features = list(train.columns).copy()

drop_features = minmax_features + minmax_neg_features + [target]
for column in drop_features:
    std_features.remove(column)

print('std_features')
display(std_features)


# TODO: Add one-feature fit Scaler, which fits on volume fraction 
# and applied to init_volume_fraction and volume_fraction
ct = ColumnTransformer(
    [
        ('minmax', MinMaxScaler(), minmax_features),
        (
            'minmax_neg', 
            MinMaxScaler(feature_range=(-1,1)), 
            minmax_neg_features
        ),
        ('std', StandardScaler(), std_features),
    ],
)
display(ct)

std_features


['sedimentation_Re', 'relative_roughness', 'K']

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ['inclination', 'volume_fraction']),
                                ('minmax_neg',
                                 MinMaxScaler(feature_range=(-1, 1)),
                                 ['wettability']),
                                ('std', StandardScaler(),
                                 ['sedimentation_Re', 'relative_roughness',
                                  'K'])])

In [8]:
# Wrapper for Statsmodel
class StatsModelsEstimator(BaseEstimator):
    def __init__(self, model_class, **init_params):
        self.model_class = model_class
        self.init_params = init_params
        
    def fit(self, X, y, **fit_params):
        self.model_ = self.model_class(endog = y, exog = X, **self.init_params)
        # Get fit_method. Pass "fit", if fit_method did not specified
        fit_method = fit_params.pop("fit_method", "fit")
        # getattr - retrieve proper method with name `fit_method`, 
        # Then, apply this method with **fit_params
        self.results_ = getattr(self.model_, fit_method)(**fit_params)
        return self

    def predict(self, X, **predict_params):
        return self.results_.predict(exog = X, **predict_params)


# Custom transformer to convert NumPy array to DataFrame with feature names
class DataFrameTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, feature_names, add_const=True):
        self.feature_names = feature_names
        self.add_const = add_const
    
    def fit(self, X, y=None):
        return self  # Nothing to fit here
    
    def transform(self, X):
        if self.add_const:
            X = sm.add_constant(X)
            columns = ['const']+self.feature_names
        else:
            columns = self.feature_names
        # Convert the NumPy array back into a DataFrame with the feature names
        return pd.DataFrame(X, columns=columns)
    

# Function to extract feature names after ColumnTransformer
def get_feature_names(column_transformer):
    # List to store the final feature names
    feature_names = []

    # Loop through each transformer in the ColumnTransformer
    for name, transformer, columns in column_transformer.transformers:
        if hasattr(transformer, 'get_feature_names_out'):
            # If transformer supports get_feature_names_out (e.g., OneHotEncoder)
            feature_names.extend(transformer.get_feature_names_out(columns))
        else:
            # Otherwise, just use the original column names (e.g., for StandardScaler)
            feature_names.extend(columns)
    
    return feature_names

feature_names = get_feature_names(ct)

In [9]:
X = train.drop(target, axis=1)
y = train[target].reset_index(drop=True)

estimator = StatsModelsEstimator(Logit)

df_transformer = DataFrameTransformer(
    feature_names, 
    add_const=False,
)

pipe = Pipeline(
    [
        ('column_transformer', ct),
        ('to_dataframe', df_transformer),
        ('model', estimator),
    ]
)

# pipe.fit(X, y, model__fit_method='fit_regularized')
pipe.fit(X, y)

print(estimator.results_.summary())

Optimization terminated successfully.
         Current function value: 0.393846
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              splashing   No. Observations:                  297
Model:                          Logit   Df Residuals:                      291
Method:                           MLE   Df Model:                            5
Date:                Fri, 04 Oct 2024   Pseudo R-squ.:                  0.3937
Time:                        00:40:08   Log-Likelihood:                -116.97
converged:                       True   LL-Null:                       -192.93
Covariance Type:            nonrobust   LLR p-value:                 5.202e-31
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
inclination            2.8884      0.627      4.604      0.000       1.659       4.118
volum

In [17]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

y_pred_proba = pipe.predict(
    X=test.drop(target, axis=1)
)

def get_prediction(y_pred_proba):
    y_pred = np.zeros_like(y_pred_proba)
    y_pred[y_pred_proba>0.5] = 1
    
    return y_pred

y_pred = get_prediction(y_pred_proba)

f1_score(test[target], y_pred)

0.8484848484848485

In [18]:
accuracy_score(test[target], y_pred)

0.8

In [19]:
roc_auc_score(
    test[target],
    y_pred_proba
)

0.8796296296296297

Idea: Consider log(relative_roughness) - DONE!

## Sedimentation Re

In [ ]:
X = train.drop(target, axis=1)
y = train[target].reset_index(drop=True)

estimator = StatsModelsEstimator(Logit)

df_transformer = DataFrameTransformer(
    feature_names, 
    add_const=True,
)

pipe = Pipeline(
    [
        ('column_transformer', ct),
        ('to_dataframe', df_transformer),
        ('model', estimator),
    ]
)

# pipe.fit(X, y, model__fit_method='fit_regularized')
pipe.fit(X, y)

print(estimator.results_.summary())

Optimization terminated successfully.
         Current function value: 0.388476
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              splashing   No. Observations:                  297
Model:                          Logit   Df Residuals:                      293
Method:                           MLE   Df Model:                            3
Date:                Fri, 04 Oct 2024   Pseudo R-squ.:                  0.4020
Time:                        00:00:36   Log-Likelihood:                -115.38
converged:                       True   LL-Null:                       -192.93
Covariance Type:            nonrobust   LLR p-value:                 2.079e-33
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                1.3957      0.216      6.472      0.000       0.973       1.818
wettability

## Volume fraction importance

In [8]:
X = train.drop(target, axis=1)
y = train[target].reset_index(drop=True)

estimator = StatsModelsEstimator(Logit)

df_transformer = DataFrameTransformer(
    feature_names, 
    add_const=True,
)

pipe = Pipeline(
    [
        ('column_transformer', ct),
        ('to_dataframe', df_transformer),
        ('model', estimator),
    ]
)

# pipe.fit(X, y, model__fit_method='fit_regularized')
pipe.fit(X, y)

print(estimator.results_.summary())

Optimization terminated successfully.
         Current function value: 0.438787
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              splashing   No. Observations:                  297
Model:                          Logit   Df Residuals:                      294
Method:                           MLE   Df Model:                            2
Date:                Thu, 03 Oct 2024   Pseudo R-squ.:                  0.3245
Time:                        23:51:40   Log-Likelihood:                -130.32
converged:                       True   LL-Null:                       -192.93
Covariance Type:            nonrobust   LLR p-value:                 6.415e-28
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               1.1459      0.186      6.168      0.000       0.782       1.510
volume_fractio

In [ ]:
X = train.drop(target, axis=1)
y = train[target].reset_index(drop=True)

estimator = StatsModelsEstimator(Logit)

df_transformer = DataFrameTransformer(
    feature_names, 
    add_const=True,
)

pipe = Pipeline(
    [
        ('column_transformer', ct),
        ('to_dataframe', df_transformer),
        ('model', estimator),
    ]
)

# pipe.fit(X, y, model__fit_method='fit_regularized')
pipe.fit(X, y)

print(estimator.results_.summary())

Optimization terminated successfully.
         Current function value: 0.433956
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              splashing   No. Observations:                  297
Model:                          Logit   Df Residuals:                      293
Method:                           MLE   Df Model:                            3
Date:                Thu, 03 Oct 2024   Pseudo R-squ.:                  0.3320
Time:                        23:52:49   Log-Likelihood:                -128.88
converged:                       True   LL-Null:                       -192.93
Covariance Type:            nonrobust   LLR p-value:                 1.390e-27
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    1.1387      0.185      6.168      0.000       0.777       1.501